# Udeco 独立边沿检测与单样本分解

本 Notebook 对 Udeco2/Udeco3 独立运行候选检测、片段合并、分段拟合、分类和分解。
`edge_signal = clean - Bcontext` 只用于 Haar/导数 primitive candidate；同方向短片段先在 `clean` 上检查连续变化并合并，边界和平台随后在 `clean` 上拟合。
attach/detach 由平台差的符号确定；recording 级幅度聚类只把明显的小幅簇降级为 ambiguous，Contact 状态不筛除边沿。

In [ ]:
# 样本与协议
SAMPLE_NAME = "cgx_rope_1"
DATA_ROOT = "subject_data"
FS = 100

# 保持原版 3 点短中值滤波；如原协议使用 4 点，可在此设置。
MEDIAN_FILTER_POINTS = 3

# 检测上下文与高召回 Haar / derivative primitive candidates
CONTEXT_WINDOW_SEC = 10.0
CONTEXT_ROBUST_ITERS = 1
EDGE_SCALES_SEC = (0.02, 0.04, 0.08, 0.16, 0.32, 0.64, 1.28, 2.56)
EDGE_Z_THRESHOLD = 2.5
EDGE_SINGLE_SCALE_Z_THRESHOLD = 3.5
EDGE_DERIVATIVE_Z_THRESHOLD = 3.5
EDGE_MIN_SCALE_SUPPORT = 2
EDGE_MERGE_SEC = 0.08

# 同方向短片段合并；不做平台检测
EDGE_FRAGMENT_MERGE_SEC = 0.02
EDGE_FRAGMENT_MIN_SPEARMAN = 0.50
EDGE_FRAGMENT_MIN_FORWARD_FRACTION = 0.60

# 在相邻 merged candidate 之间、于 clean 上做完整 morphology 拟合
EDGE_FIT_MIN_WINDOW_SEC = 0.25
EDGE_FIT_MAX_WINDOW_SEC = 1.50
EDGE_MIN_SNR = 3.0
EDGE_MIN_FIT_IMPROVEMENT = 0.15
EDGE_MIN_PLATEAU_CONTRAST = 0.50
AMPLITUDE_CLUSTER_MIN_SEPARATION = 2.50

# 后续分解参数；不参与边沿是否成立的决定
UNCERTAIN_MARGIN_SEC = 0.05
CONTACT_ANCHOR_WINDOW_SEC = 1.0
CONTACT_LOWESS_WINDOW_SEC = 3.0
CONTACT_ROBUST_ITERS = 2
BSLOW_TAU_SEC = 60.0
BSLOW_UPDATE_RATE = 1.0
BSLOW_STABLE_SLOPE_THRESHOLD = 3e-4
BSLOW_STABLE_OFFSET_THRESHOLD = 1e-3
PRESSURE_SLOPE_THRESHOLD = 3e-4
PRESSURE_OFFSET_THRESHOLD = 1e-3
J_SMOOTH_WINDOW_SEC = 0.10

# 交互绘图
PLOT_START_SEC = 70.0
PLOT_END_SEC = 140.0
LINK_X_AXES_WITHIN_CHANNEL = True
EDGE_TABLE_MAX_ROWS = 80

## 导入与数据读取

`Bcontext` 只负责候选检测。最终 `Bm_contact` 会在接触模态重建后重新估计。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from udeco_analysis import load_udeco_sample
from udeco_edge_detection import (
    CascadeStateConfig,
    decompose_cascade_udeco,
)

# plotly_mimetype 会把交互图的 JSON 保存在 Notebook 输出中，重新打开后仍可缩放和悬停。
pio.renderers.default = "plotly_mimetype"

In [ ]:
# 在所有受试者目录下唯一匹配传感器 CSV；因此切换样本只需改首个代码单元的 SAMPLE_NAME。
data_root = Path(DATA_ROOT).resolve()
matches = sorted(data_root.glob(f"*/{SAMPLE_NAME}_sensor.csv"))
if len(matches) != 1:
    raise FileNotFoundError(
        f"期望唯一匹配 */{SAMPLE_NAME}_sensor.csv，实际找到 {len(matches)} 个：{matches}"
    )

loaded = load_udeco_sample(matches[0], data_root=data_root, fs=FS)
time = loaded.time_s
udeco2 = loaded.udeco2
udeco3 = loaded.udeco3
fs = loaded.fs

print(f"样本：{loaded.sample_name}（{loaded.subject}）")
print(f"文件：{matches[0]}")
print(f"采样率：{fs} Hz；Udeco2/Udeco3 长度：{len(udeco2)}/{len(udeco3)}")

## 双通道独立分解

下面只构造一次配置，并对 Udeco2/Udeco3 分别调用同一接口。两次调用之间不共享局部噪声、候选事件、状态或 Bslow。

In [ ]:
config = CascadeStateConfig(
    median_filter_points=MEDIAN_FILTER_POINTS,
    context_window_s=CONTEXT_WINDOW_SEC,
    context_robust_iterations=CONTEXT_ROBUST_ITERS,
    edge_scales_s=EDGE_SCALES_SEC,
    edge_z_threshold=EDGE_Z_THRESHOLD,
    edge_single_scale_z_threshold=EDGE_SINGLE_SCALE_Z_THRESHOLD,
    edge_derivative_z_threshold=EDGE_DERIVATIVE_Z_THRESHOLD,
    edge_min_scale_support=EDGE_MIN_SCALE_SUPPORT,
    edge_merge_s=EDGE_MERGE_SEC,
    edge_fragment_merge_s=EDGE_FRAGMENT_MERGE_SEC,
    edge_fragment_min_spearman=EDGE_FRAGMENT_MIN_SPEARMAN,
    edge_fragment_min_forward_fraction=EDGE_FRAGMENT_MIN_FORWARD_FRACTION,
    edge_fit_min_window_s=EDGE_FIT_MIN_WINDOW_SEC,
    edge_fit_max_window_s=EDGE_FIT_MAX_WINDOW_SEC,
    edge_min_snr=EDGE_MIN_SNR,
    edge_min_fit_improvement=EDGE_MIN_FIT_IMPROVEMENT,
    edge_min_plateau_contrast=EDGE_MIN_PLATEAU_CONTRAST,
    amplitude_cluster_min_separation=AMPLITUDE_CLUSTER_MIN_SEPARATION,
    uncertain_margin_s=UNCERTAIN_MARGIN_SEC,
    contact_anchor_window_s=CONTACT_ANCHOR_WINDOW_SEC,
    contact_lowess_window_s=CONTACT_LOWESS_WINDOW_SEC,
    contact_robust_iterations=CONTACT_ROBUST_ITERS,
    bslow_tau_s=BSLOW_TAU_SEC,
    bslow_update_rate=BSLOW_UPDATE_RATE,
    bslow_stable_slope_threshold=BSLOW_STABLE_SLOPE_THRESHOLD,
    bslow_stable_offset_threshold=BSLOW_STABLE_OFFSET_THRESHOLD,
    pressure_slope_threshold=PRESSURE_SLOPE_THRESHOLD,
    pressure_offset_threshold=PRESSURE_OFFSET_THRESHOLD,
    j_smooth_window_s=J_SMOOTH_WINDOW_SEC,
)

result2 = decompose_cascade_udeco(udeco2, fs, config)
result3 = decompose_cascade_udeco(udeco3, fs, config)

BASELINE_COUNTS = {
    "Udeco2": {"primitive": 2536, "event": 538},
    "Udeco3": {"primitive": 2430, "event": 537},
}
for name, result in (("Udeco2", result2), ("Udeco3", result3)):
    baseline = BASELINE_COUNTS[name]
    noncontact_s = np.count_nonzero(result["noncontact_mask"]) / fs
    print(
        f"{name}: 基线 {baseline['primitive']} primitives / {baseline['event']} events；"
        f"当前 {result['primitive_count']} primitives → {result['merged_count']} merged → "
        f"{result['morphology_event_count']} morphology 合格 → "
        f"{result['post_amplitude_event_count']} 幅度过滤后 events；"
        f"Non-contact {noncontact_s:.2f} s"
    )

## 候选边沿、片段合并与局部质量

每个 merged candidate 只在 `clean` 上做一次完整 morphology 拟合。`localization_scale` 负责定位 center，`morphology_scale` 与完整 merged span 决定拟合窗口。
表中 `start/end` 是拟合的 0%/100% 点；`edge_snr`、拟合改善、平台斜率对比和单调性显示局部证据。幅度 K=2 仅在两个 log-amplitude 簇明显分离时启用，小幅簇保留为 `ambiguous`。
`J_events` 包含最终成立的边沿，状态掩码只供后续分解使用，不会反向删除边沿。

In [ ]:
CANDIDATE_COLUMNS = [
    "center_s", "start_s", "end_s", "edge_width_points", "edge_width_s",
    "primitive_count", "fragment_centers", "merged_start_index", "merged_end_index",
    "localization_scale", "morphology_scale", "best_scale_s",
    "delta", "left_level", "right_level", "left_mad", "right_mad",
    "noise_left", "noise_right", "edge_snr", "fit_residual",
    "fit_improvement", "plateau_contrast", "monotonicity",
    "detection_channels", "scale_support", "support_scales_s",
    "pre_amplitude_candidate_type", "amplitude_cluster",
    "candidate_type", "confidence", "accepted",
    "classification_reasons", "rejection_reason",
]


def candidate_table(result):
    '''将内部索引整理成便于人工核验的秒制候选表。'''
    rows = []
    for item in result["edge_candidates"]:
        row = dict(item)
        row["center_s"] = row.get("center_index", 0) / fs
        row["start_s"] = row["start_index"] / fs if row.get("start_index") is not None else np.nan
        row["end_s"] = row["end_index"] / fs if row.get("end_index") is not None else np.nan
        row["detection_channels"] = ", ".join(row.get("detection_channels", ()))
        row["classification_reasons"] = ", ".join(row.get("classification_reasons", ()))
        rows.append(row)
    frame = pd.DataFrame(rows)
    if frame.empty:
        return pd.DataFrame(columns=CANDIDATE_COLUMNS)
    return frame.reindex(columns=CANDIDATE_COLUMNS).sort_values("center_s")


def event_table(result):
    '''展示幅度过滤后仍成立的 attach/detach。'''
    frame = candidate_table({"edge_candidates": result["J_events"]})
    if frame.empty:
        return frame
    frame.insert(0, "event_type", [item["event_type"] for item in result["J_events"]])
    return frame


def print_edge_filter_stats(name, result):
    items = result["edge_candidates"]
    merged_examples = [item for item in items if item.get("primitive_count", 1) > 1]
    small_examples = [item for item in items if item.get("amplitude_cluster") == "small"]
    print(
        f"{name}：{result['primitive_count']} primitives → {result['merged_count']} merged；"
        f"{result['morphology_event_count']} morphology 合格 → {len(result['J_events'])} 成立边沿"
    )
    print(
        f"  多片段合并组 {len(merged_examples)}；小幅簇降级 {len(small_examples)}；"
        f"幅度簇分离={result['amplitude_clusters_separated']} "
        f"(score={result['amplitude_cluster_separation']:.2f})"
    )
    print(pd.Series([item["candidate_type"] for item in items], dtype="string").value_counts().to_string())
    reasons = [reason for item in items for reason in item.get("classification_reasons", ())]
    print("局部或幅度证据不足原因：")
    counts = pd.Series(reasons, dtype="string").value_counts()
    print("\n".join(f"  {reason}: {count}" for reason, count in counts.items()) or "  无")


for name, result in (("Udeco2", result2), ("Udeco3", result3)):
    candidates = candidate_table(result)
    print_edge_filter_stats(name, result)
    print(f"\n{name} 候选边沿：共 {len(candidates)} 行，显示前 {EDGE_TABLE_MAX_ROWS} 行")
    display(candidates.head(EDGE_TABLE_MAX_ROWS))
    print(f"{name} 连续 fragment 合并案例：")
    display(candidates[candidates["primitive_count"].fillna(1) > 1].head(12))
    print(f"{name} 小幅簇降级案例：")
    display(candidates[candidates["amplitude_cluster"] == "small"].head(12))
    print(f"{name} 幅度过滤后成立边沿：")
    display(event_table(result))

## 交互诊断图

每路维持独立的六行图。第三行在 `clean` 上标注拟合的起止点；第四行同时展示 Haar 与导数得分。
浅色 Contact/Non-contact 背景是依据最近边沿推断的后续分解状态，不参与边沿筛除。

In [ ]:
EDGE_COLORS = {
    "detach": "#d62728",
    "attach": "#1f77b4",
    "ambiguous": "#7f7f7f",
}


def _plot_range(time_axis):
    '''计算初始显示范围；只改变坐标轴，不裁剪算法数组。'''
    data_start, data_end = float(time_axis[0]), float(time_axis[-1])
    start = data_start if PLOT_START_SEC is None else max(float(PLOT_START_SEC), data_start)
    end = data_end if PLOT_END_SEC is None else min(float(PLOT_END_SEC), data_end)
    return [data_start, data_end] if start >= end else [start, end]


def _mask_spans(mask, time_axis, min_points=1, merge_gap_points=0, max_spans=160):
    '''把布尔 mask 压缩为绘图矩形区间，避免逐点添加 shape。'''
    indices = np.flatnonzero(np.asarray(mask, dtype=bool))
    if not indices.size:
        return []
    breaks = np.flatnonzero(np.diff(indices) > merge_gap_points + 1) + 1
    groups = np.split(indices, breaks)
    dt = 1.0 / fs
    spans = [
        (float(time_axis[group[0]]), float(time_axis[group[-1]] + dt))
        for group in groups if len(group) >= min_points
    ]
    return spans[:max_spans]


def _add_mask_regions(fig, mask, time_axis, row, color, opacity, min_points=1, merge_gap=0):
    '''在指定子图添加状态/更新区背景；背景不修改任何数据。'''
    for x0, x1 in _mask_spans(mask, time_axis, min_points, merge_gap):
        fig.add_vrect(
            x0=x0, x1=x1, fillcolor=color, opacity=opacity,
            line_width=0, layer="below", row=row, col=1,
        )


def _add_state_regions(fig, result, time_axis, row):
    '''以互不覆盖的颜色显示推断状态；Uncertain 叠在最上层。'''
    _add_mask_regions(fig, result["contact_mask"], time_axis, row, "#2ca02c", 0.045, 3, 2)
    _add_mask_regions(fig, result["noncontact_mask"], time_axis, row, "#1f77b4", 0.12, 1, 0)
    _add_mask_regions(fig, result["transition_mask"], time_axis, row, "#ff7f0e", 0.16, 1, 0)
    _add_mask_regions(fig, result["uncertain_mask"], time_axis, row, "#7f7f7f", 0.10, 3, 2)


def _candidate_customdata(items):
    '''构造 Plotly 悬停信息；缺失字段用 NaN/空字符串清楚呈现。'''
    return np.asarray([
        [
            item.get("edge_width_points", np.nan),
            item.get("edge_width_s", np.nan),
            item.get("fit_improvement", np.nan),
            item.get("edge_snr", np.nan),
            item.get("direction", ""),
            item.get("candidate_type", ""),
            item.get("confidence", np.nan),
            item.get("rejection_reason") or "accepted",
            ", ".join(item.get("detection_channels", ())),
            ", ".join(item.get("classification_reasons", ())),
            item.get("primitive_count", 1),
            item.get("localization_scale", np.nan),
            item.get("morphology_scale", np.nan),
            item.get("amplitude_cluster", ""),
        ]
        for item in items
    ], dtype=object)


def plot_channel_diagnostics(time_axis, result, channel_name, sample_name):
    '''绘制单路六行交互诊断图；两路分别调用，因此可独立放大。'''
    fig = make_subplots(
        rows=6, cols=1,
        shared_xaxes=LINK_X_AXES_WITHIN_CHANNEL,
        vertical_spacing=0.028,
        specs=[[{}], [{"secondary_y": True}], [{}], [{}], [{}], [{}]],
        subplot_titles=(
            "1. Raw / 短窗 Median clean",
            "2. 低频 Bcontext / 去上下文 edge_signal",
            "3. clean 上拟合的 ambiguous / attach / detach 起止点",
            "4. Haar / 导数候选与推断状态",
            "5. 接触模态重建 Ucontact / 最终 Bm_contact",
            "6. Bslow / P / J（绿色背景：允许 Bslow 更新）",
        ),
    )

    fig.add_trace(go.Scatter(x=time_axis, y=result["raw"], name="Raw", line=dict(color="#555", width=1)), row=1, col=1)
    fig.add_trace(go.Scatter(x=time_axis, y=result["clean"], name="Median clean", line=dict(color="#9467bd", width=1)), row=1, col=1)

    fig.add_trace(go.Scatter(x=time_axis, y=result["Bcontext"], name="Bcontext (10 s)", line=dict(color="#2ca02c", width=2)), row=2, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(x=time_axis, y=result["edge_signal"], name="clean-Bcontext", line=dict(color="#7f7f7f", width=1)), row=2, col=1, secondary_y=True)

    # 在 clean 上标注分段拟合边界；细线连起止点，符号区分起点/终点。
    fig.add_trace(go.Scatter(x=time_axis, y=result["clean"], name="clean / candidate intervals", line=dict(color="#555", width=1)), row=3, col=1)
    for kind in ("ambiguous", "attach", "detach"):
        items = [item for item in result["edge_candidates"] if item.get("candidate_type") == kind]
        # 无可靠前/后平台时只画中心点，不把中心伪装成 start/end。
        unresolved = [item for item in items if item.get("start_index") is None or item.get("end_index") is None]
        if unresolved:
            centers = np.asarray([int(item["center_index"]) for item in unresolved])
            fig.add_trace(go.Scatter(
                x=time_axis[centers], y=result["clean"][centers], mode="markers",
                name=f"{kind} 中心（边界未定）", marker=dict(color=EDGE_COLORS[kind], size=8, symbol="circle-open"),
                customdata=_candidate_customdata(unresolved),
                hovertemplate="%{x:.3f} s / clean=%{y:.5f}<br>边界未定<br>形态原因=%{customdata[9]}<extra></extra>",
            ), row=3, col=1)
        items = [item for item in items if item.get("start_index") is not None and item.get("end_index") is not None]
        if not items:
            continue
        starts = np.asarray([int(item.get("start_index", item["center_index"])) for item in items])
        ends = np.asarray([int(item.get("end_index", item["center_index"])) for item in items])
        starts = np.clip(starts, 0, len(time_axis) - 1)
        ends = np.clip(ends, 0, len(time_axis) - 1)
        segment_x = [point for start, end in zip(starts, ends) for point in (time_axis[start], time_axis[end], None)]
        segment_y = [point for start, end in zip(starts, ends) for point in (result["clean"][start], result["clean"][end], None)]
        fig.add_trace(go.Scatter(x=segment_x, y=segment_y, mode="lines", showlegend=False, hoverinfo="skip", line=dict(color=EDGE_COLORS[kind], width=1.4)), row=3, col=1)
        for phase, indices, symbol in (("起点", starts, "triangle-up"), ("终点", ends, "triangle-down")):
            fig.add_trace(go.Scatter(
                x=time_axis[indices], y=result["clean"][indices], mode="markers",
                name=f"{kind} {phase}", marker=dict(color=EDGE_COLORS[kind], size=7, symbol=symbol),
                customdata=_candidate_customdata(items),
                hovertemplate=("%{x:.3f} s / clean=%{y:.5f}<br>" + kind + " " + phase + "<br>"
                               "完整宽度=%{customdata[0]} 点 / %{customdata[1]:.4f} s；拟合改善=%{customdata[2]:.2f}<br>"
                               "检测通道=%{customdata[8]}；局部 SNR=%{customdata[3]:.2f}<br>方向=%{customdata[4]}<br>"
                               "片段数=%{customdata[10]}；定位/形态尺度=%{customdata[11]:.3f}/%{customdata[12]:.3f} s<br>"
                               "幅度簇=%{customdata[13]}；置信度=%{customdata[6]:.2f}<br>结果=%{customdata[7]}<extra></extra>"),
            ), row=3, col=1)

    _add_state_regions(fig, result, time_axis, row=4)
    fig.add_trace(go.Scatter(x=time_axis, y=result["edge_score"], name="edge_score", line=dict(color="#1f77b4", width=1)), row=4, col=1)
    fig.add_trace(go.Scatter(x=time_axis, y=result["derivative_score"], name="derivative_score", line=dict(color="#ff7f0e", width=1)), row=4, col=1)
    fig.add_hline(y=EDGE_Z_THRESHOLD, line_dash="dash", line_color="#d62728", row=4, col=1)
    fig.add_hline(y=-EDGE_Z_THRESHOLD, line_dash="dash", line_color="#d62728", row=4, col=1)
    for kind in ("detach", "attach", "ambiguous"):
        items = [item for item in result["edge_candidates"] if item.get("candidate_type") == kind]
        if not items:
            continue
        centers = np.asarray([int(item.get("center_index", 0)) for item in items])
        centers = np.clip(centers, 0, len(time_axis) - 1)
        fig.add_trace(
            go.Scatter(
                x=time_axis[centers], y=[result["derivative_score"][c] if "derivative" in item.get("detection_channels", ()) else result["edge_score"][c] for item, c in zip(items, centers)],
                mode="markers", name=f"candidate: {kind}",
                marker=dict(color=EDGE_COLORS[kind], size=8, symbol="diamond"),
                customdata=_candidate_customdata(items),
                hovertemplate=(
                    "t=%{x:.3f} s<br>candidate score=%{y:.2f}<br>"
                    "完整宽度=%{customdata[0]} 点 / %{customdata[1]:.4f} s；拟合改善=%{customdata[2]:.2f}<br>"
                    "检测通道=%{customdata[8]}；局部 SNR=%{customdata[3]:.2f}<br>方向=%{customdata[4]}<br>"
                    "分类=%{customdata[5]}<br>置信度=%{customdata[6]:.2f}<br>"
                    "片段数=%{customdata[10]}；定位/形态尺度=%{customdata[11]:.3f}/%{customdata[12]:.3f} s<br>"
                    "幅度簇=%{customdata[13]}；形态原因=%{customdata[9]}<br>结果=%{customdata[7]}<extra></extra>"
                ),
            ), row=4, col=1,
        )

    _add_mask_regions(fig, result["ucontact_synthetic_mask"], time_axis, 5, "#17becf", 0.13, 1, 0)
    fig.add_trace(go.Scatter(x=time_axis, y=result["Ucontact"], name="Ucontact", line=dict(color="#17becf", width=1.4)), row=5, col=1)
    fig.add_trace(go.Scatter(x=time_axis, y=result["Bm_contact"], name="Bm_contact", line=dict(color="#ff7f0e", width=2)), row=5, col=1)

    # update mask 往往断续，合并极短间隙并过滤不足 0.1 s 的片段，让绿色背景可读。
    _add_mask_regions(fig, result["bslow_update_mask"], time_axis, 6, "#2ca02c", 0.09, max(2, int(0.10 * fs)), 3)
    fig.add_trace(go.Scatter(x=time_axis, y=result["Bslow"], name="Bslow", line=dict(color="#2ca02c", width=2)), row=6, col=1)
    fig.add_trace(go.Scatter(x=time_axis, y=result["P"], name="P", line=dict(color="#d62728", width=1.5)), row=6, col=1)
    fig.add_trace(go.Scatter(x=time_axis, y=result["J"], name="J", line=dict(color="#9467bd", width=1.5)), row=6, col=1)

    for row in range(1, 7):
        fig.update_xaxes(range=_plot_range(time_axis), title_text="Time (s)" if row == 6 else None, row=row, col=1)
        fig.update_yaxes(title_text="Udeco" if row != 4 else "Robust z-score", row=row, col=1)
    fig.update_yaxes(title_text="clean-Bcontext", row=2, col=1, secondary_y=True)
    fig.update_layout(
        title=f"{sample_name} — {channel_name} 独立边沿与分解诊断",
        height=1500, template="plotly_white", hovermode="x unified",
        dragmode="zoom", legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
        margin=dict(l=70, r=70, t=125, b=55),
    )
    fig.show(config={"scrollZoom": True, "displaylogo": False})
    return fig


def plot_summary(time_axis, result_u2, result_u3, sample_name):
    '''绘制最终双路摘要；两行 x 轴不联动，便于分别放大核验。'''
    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=False, vertical_spacing=0.10,
        subplot_titles=("Udeco2: Raw / clean / J / P / Bslow / residual", "Udeco3: Raw / clean / J / P / Bslow / residual"),
    )
    for row, result in enumerate((result_u2, result_u3), start=1):
        _add_state_regions(fig, result, time_axis, row=row)
        fig.add_trace(go.Scatter(x=time_axis, y=result["raw"], name=f"Raw Udeco{row + 1}", line=dict(color="#555", width=1)), row=row, col=1)
        # clean 是进入分解算法的 3/4 点短中值滤波信号；与 Raw 同图可检查边沿是否被钝化。
        fig.add_trace(go.Scatter(x=time_axis, y=result["clean"], name=f"clean Udeco{row + 1}", line=dict(color="#17becf", width=1.3)), row=row, col=1)
        fig.add_trace(go.Scatter(x=time_axis, y=result["J"], name=f"J{row + 1}", line=dict(color="#9467bd", width=1.5)), row=row, col=1)
        fig.add_trace(go.Scatter(x=time_axis, y=result["P"], name=f"P{row + 1}", line=dict(color="#d62728", width=1.5)), row=row, col=1)
        fig.add_trace(go.Scatter(x=time_axis, y=result["Bslow"], name=f"Bslow{row + 1}", line=dict(color="#2ca02c", width=2)), row=row, col=1)
        # residual 是 clean 扣除 J、P、Bslow 后的剩余项，不是 Raw 的滤波误差。
        fig.add_trace(go.Scatter(x=time_axis, y=result["residual"], name=f"residual{row + 1}", line=dict(color="#8c564b", width=1, dash="dot")), row=row, col=1)
        fig.update_xaxes(range=_plot_range(time_axis), title_text="Time (s)", row=row, col=1)
        fig.update_yaxes(title_text="Udeco", row=row, col=1)
    fig.update_layout(
        title=f"{sample_name} — Udeco 级联状态最终分解",
        height=800, template="plotly_white", hovermode="x unified", dragmode="zoom",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
        margin=dict(l=70, r=40, t=120, b=55),
    )
    fig.show(config={"scrollZoom": True, "displaylogo": False})
    return fig

In [ ]:
# Udeco2 独立交互图：缩放和平移不会改变 Udeco3 的视图。
diagnostic_fig2 = plot_channel_diagnostics(time, result2, "Udeco2", SAMPLE_NAME)

In [ ]:
# Udeco3 独立交互图：重点核验其候选是否被正确保留为压力或慢变，而非伪 Non-contact。
diagnostic_fig3 = plot_channel_diagnostics(time, result3, "Udeco3", SAMPLE_NAME)

## 最终分解摘要

最后一个代码单元只负责显示最终结果。上下两图分别为 Udeco2/Udeco3，每图同时显示 Raw、clean、J、P、Bslow、residual，并叠加 Contact/Non-contact/Transition/Uncertain 状态背景；两行可以分别放大。Raw 未经中值滤波；分解恒等式对应 clean = J + P + Bslow + residual。各分量需要按同一时刻相加，不能期待它们在图上逐条重合。

In [ ]:
summary_fig = plot_summary(time, result2, result3, SAMPLE_NAME)